In [ ]:
import torch
import numpy as np
import scanpy as sc
import pandas as pd
import numpy as np, torch, scipy.sparse as sp
import torch
import scvi

import sys 
sys.path.append("/home/zhouyj/stVirtual/src")
from stVirtual_model.lr_dec import *

# Train a decoder for LR gene

In [ ]:
base_dir = '/home/zhouyj/stVirtual/data/Mbrain/processed'

adata = sc.read_h5ad(f'{base_dir}/Mbrain.h5ad')

adata.obs_names_make_unique()

In [ ]:
adata.obs["annotation"] = adata.obs["annotation"].astype("category")
adata.obs["annotation"] = adata.obs["annotation"].cat.remove_unused_categories()

print(adata.obs["annotation"].unique())

lr_pairs = pd.read_csv("/home/zhouyj/stVirtual/data/lrpairs/mouse/LR_pairs.csv")
print(lr_pairs.iloc[:1, 2:4])
ligand_list = lr_pairs['ligand'].unique()
receptor_list = lr_pairs['receptor'].unique()

gene_idx = pd.Index(adata.var_names.astype(str))
ligand   = gene_idx.intersection(pd.Index(ligand_list))
receptor = gene_idx.intersection(pd.Index(receptor_list))
ligand_df   = pd.DataFrame(ligand.sort_values(), columns=["ligand"])
receptor_df = pd.DataFrame(receptor.sort_values(), columns=["receptor"])
print("ligand gene num: ", len(ligand))
print("receptor gene num: ", len(receptor))
feat = adata.var_names.astype(str).values

X = adata.layers['counts']
X = X.toarray() if sp.issparse(X) else X

expr = pd.DataFrame(X, index=adata.obs_names, columns=feat)

expr_L = expr.loc[:, expr.columns.intersection(ligand)]
expr_R = expr.loc[:, expr.columns.intersection(receptor)]

print(expr_L.iloc[:2, :2])
print(expr_R.iloc[:2, :2])

['purkinje layer', 'granular layer', 'molecular layer', 'white matter']
Categories (4, object): ['granular layer', 'molecular layer', 'purkinje layer', 'white matter']
  ligand   receptor
0  Tgfb1  TGFbR1_R2
ligand gene num:  378
receptor gene num:  295
         Col9a1  Sema4c
268_226     0.0     0.0
217_190     0.0     0.0
         Npbwr1  Oprk1
268_226     0.0    0.0
217_190     0.0    0.0


In [ ]:
model_dir = "/home/zhouyj/stVirtual/src/Mbrain/Mbrain_ckpt/scvi/scanvi"
model = scvi.model.SCANVI.load(model_dir, adata)

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.


INFO     File /home/zhouyj/project/src/mouse_ckpt copy/scvi/scanvi/model.pt already downloaded                     


## TRAIN DECORDER

In [ ]:
OUT_DIR = "/home/zhouyj/stVirtual/src/Mbrain/Mbrain_ckpt/exp_decoder"
SAMPLE_IDS = ["T167", "T168", "T169", "T170", "T171"]

genes = sorted(set(map(str, ligand)) | set(map(str, receptor)))

results = train_count_decoder_pairs(
    adata=adata,
    model=model,
    genes=genes,
    sample_ids=SAMPLE_IDS,
    sample_key="sample",
    out_dir=OUT_DIR,
    sample_prefix="decoder_counts_pair",
    layer_counts="counts",
    skip_if_exists=False,
    seed=2025,
    test_size=0.2,
    batch_size=512,
    epochs=200,
    patience=10,
    lr=1e-3,
    device="cuda:0",
)
for r in results:
    print(r)


INFO     AnnData object appears to be a copy. Attempting to transfer setup.                                        


T167_T168 ep 088/200:  44%|███▉     | 87/200 [00:27<00:35,  3.18it/s, tr=0.0419 va=0.0421]


INFO     AnnData object appears to be a copy. Attempting to transfer setup.                                        


T168_T169 ep 079/200:  39%|███▌     | 78/200 [00:37<00:58,  2.07it/s, tr=0.0477 va=0.0473]


INFO     AnnData object appears to be a copy. Attempting to transfer setup.                                        


T169_T170 ep 094/200:  46%|████▏    | 93/200 [00:54<01:02,  1.72it/s, tr=0.0466 va=0.0469]


INFO     AnnData object appears to be a copy. Attempting to transfer setup.                                        


T170_T171 ep 074/200:  36%|███▎     | 73/200 [00:50<01:27,  1.45it/s, tr=0.0400 va=0.0402]

DecoderTrainResult(pair_id='T167_T168', n_cells=37692, n_genes=637, best_val=0.042112282687130534, decoder_ckpt_path='/home/zhouyj/project/src/mouse_ckpt copy/exp_decoder/decoder_counts_pair_T167_T168.pt', decoder_stat_path='/home/zhouyj/project/src/mouse_ckpt copy/exp_decoder/decoder_counts_pair_T167_T168.json')
DecoderTrainResult(pair_id='T168_T169', n_cells=57408, n_genes=637, best_val=0.047309159991257906, decoder_ckpt_path='/home/zhouyj/project/src/mouse_ckpt copy/exp_decoder/decoder_counts_pair_T168_T169.pt', decoder_stat_path='/home/zhouyj/project/src/mouse_ckpt copy/exp_decoder/decoder_counts_pair_T168_T169.json')
DecoderTrainResult(pair_id='T169_T170', n_cells=68795, n_genes=637, best_val=0.04694020735895802, decoder_ckpt_path='/home/zhouyj/project/src/mouse_ckpt copy/exp_decoder/decoder_counts_pair_T169_T170.pt', decoder_stat_path='/home/zhouyj/project/src/mouse_ckpt copy/exp_decoder/decoder_counts_pair_T169_T170.json')
DecoderTrainResult(pair_id='T170_T171', n_cells=80973, n